# 4-Channel YOLOv8 (Early Fusion) for RGB-T Drone Imagery
This notebook trains a 4-channel early-fusion YOLOv8 model.

In [1]:
!pip install ultralytics -q
import ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [2]:
import os, shutil, yaml
from pathlib import Path
from ultralytics import YOLO

MODEL_SCALE = 'n'
DROP_CLASS  = 'uncertain'

dataset_root = '/kaggle/input/datasets/deveshagarwal652/yolo-rgbt/YOLO_RGBT_4CH'
work_root    = Path('/kaggle/working/ds3')

# ---------- filtered 3-class dataset ----------
with open(f'{dataset_root}/dataset.yaml') as f:
    cfg = yaml.safe_load(f)

names = cfg['names']
names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
drop_id = names.index(DROP_CLASS)
new_names = [n for i, n in enumerate(names) if i != drop_id]
print('Classes:', names, '->', new_names)

def remap_id(c):
    return c - (1 if c > drop_id else 0)

def to_label_dir(img_dir: Path) -> Path:
    parts = list(img_dir.parts)
    idx = max(i for i, p in enumerate(parts) if p == 'images')
    parts[idx] = 'labels'
    return Path(*parts)

def build_split(rel):
    src_img = Path(dataset_root) / rel
    src_lbl = to_label_dir(src_img)
    dst_img = work_root / rel
    dst_lbl = to_label_dir(dst_img)
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    exts = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
    n_img = n_drop = 0
    for img in src_img.iterdir():
        if img.suffix.lower() not in exts:
            continue
        link = dst_img / img.name
        if not link.exists():
            os.symlink(img, link)
        n_img += 1
        lbl = src_lbl / (img.stem + '.txt')
        out = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                p = line.split()
                if not p:
                    continue
                c = int(float(p[0]))
                if c == drop_id:
                    n_drop += 1
                    continue
                out.append(' '.join([str(remap_id(c))] + p[1:]))
        (dst_lbl / (img.stem + '.txt')).write_text('\n'.join(out) + ('\n' if out else ''))
    print(f'{rel}: {n_img} images, removed {n_drop} "{DROP_CLASS}" boxes')

if work_root.exists():
    shutil.rmtree(work_root)

new_cfg = {k: v for k, v in cfg.items() if k not in ('path', 'train', 'val', 'test', 'names', 'nc')}
new_cfg['path'] = str(work_root)
new_cfg.setdefault('channels', 4)
for split in ('train', 'val', 'test'):
    if split in cfg and cfg[split]:
        rel = cfg[split] if isinstance(cfg[split], str) else cfg[split][0]
        build_split(rel)
        new_cfg[split] = rel
new_cfg['names'] = {i: n for i, n in enumerate(new_names)}

with open('/kaggle/working/dataset_3cls.yaml', 'w') as f:
    yaml.safe_dump(new_cfg, f, sort_keys=False)
print(open('/kaggle/working/dataset_3cls.yaml').read())

# ---------- 4-channel P2 model yaml ----------
with open(f'/kaggle/working/yolov8{MODEL_SCALE}-p2-4ch.yaml', 'w') as f:
    f.write(f'''
nc: {len(new_names)}
channels: 4

scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]

  - [[18, 21, 24, 27], 1, Detect, [nc]]
''')

# download pretrained weights once, so the two processes don't race
YOLO(f'yolov8{MODEL_SCALE}.pt')
print('Prep done.')

Classes: ['person', 'crowd', 'rider', 'uncertain'] -> ['person', 'crowd', 'rider']
images/train: 4900 images, removed 1340 "uncertain" boxes
images/val: 1225 images, removed 106 "uncertain" boxes
channels: 4
path: /kaggle/working/ds3
train: images/train
val: images/val
names:
  0: person
  1: crowd
  2: rider

Prep done.


In [15]:
import os, yaml
from pathlib import Path

src = Path('/kaggle/working/ds3')      # already has no 'uncertain'
dst = Path('/kaggle/working/ds1')

for split in ('train', 'val'):
    (dst / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dst / 'labels' / split).mkdir(parents=True, exist_ok=True)
    for img in (src / 'images' / split).iterdir():
        link = dst / 'images' / split / img.name
        if not link.exists():
            os.symlink(os.path.realpath(img), link)
    n = 0
    for lbl in (src / 'labels' / split).glob('*.txt'):
        out = []
        for line in lbl.read_text().splitlines():
            p = line.split()
            if p:
                out.append(' '.join(['0'] + p[1:]))   # person/crowd/rider -> 0
                n += 1
        (dst / 'labels' / split / lbl.name).write_text('\n'.join(out) + ('\n' if out else ''))
    print(split, 'boxes:', n)

for ch in (3, 4):
    with open(f'/kaggle/working/ds1_{ch}ch.yaml', 'w') as f:
        yaml.safe_dump({'path': str(dst), 'train': 'images/train', 'val': 'images/val',
                        'channels': ch, 'names': {0: 'human'}}, f, sort_keys=False)

    base = open('/kaggle/working/yolov8n-p2-4ch.yaml').read()
    base = base.replace('channels: 4', f'channels: {ch}')
    import re
    base = re.sub(r'nc: \d+', 'nc: 1', base)
    open(f'/kaggle/working/yolov8n-p2-1cls-{ch}ch.yaml', 'w').write(base)
print('done')

train boxes: 54974
val boxes: 14460
done


In [16]:
%%writefile /kaggle/working/train_1cls.py
import os, cv2, torch
import numpy as np
from ultralytics import YOLO
import ultralytics.data.base as base

MODE   = os.environ.get('MODE', 'thermal')   # 'thermal' or '4ch'
IMGSZ  = 1024
EPOCHS = 40
BATCH  = 32
NAME   = f'human_{MODE}_{EPOCHS}ep'
CH     = 3 if MODE == 'thermal' else 4

data_yaml  = f'/kaggle/working/ds1_{CH}ch.yaml'
model_yaml = f'/kaggle/working/yolov8n-p2-1cls-{CH}ch.yaml'

def imread_custom(p, flags=cv2.IMREAD_COLOR):
    img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_UNCHANGED)
    if MODE == 'thermal':
        return np.ascontiguousarray(img[:, :, [3, 3, 3]])
    return np.ascontiguousarray(img[:, :, [3, 0, 1, 2]])   # flipped later to R,G,B,T

base.imread = imread_custom

model = YOLO(model_yaml).load('yolov8n.pt')

if MODE == '4ch':
    w3 = YOLO('yolov8n.pt').model.model[0].conv.weight.data
    with torch.no_grad():
        c = model.model.model[0].conv.weight
        c[:, :3] = w3
        c[:, 3:] = w3.mean(1, keepdim=True)

model.train(
    data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device='0,1', workers=2, patience=0, close_mosaic=5,
    name=NAME, exist_ok=True,
    hsv_h=0, hsv_s=0, hsv_v=0,
    scale=0.3, mixup=0.0, flipud=0.5, fliplr=0.5,
)

if int(os.environ.get('RANK', -1)) in (-1, 0):
    best = YOLO(f'/kaggle/working/runs/detect/{NAME}/weights/best.pt')
    m = best.val(data=data_yaml, imgsz=IMGSZ, batch=16, device=0, workers=2)
    print(f'[{MODE}] P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}')

Writing /kaggle/working/train_1cls.py


In [ ]:
!MODE=4ch PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True torchrun --nproc_per_node 2 /kaggle/working/train_1cls.py

Transferred 219/437 items from pretrained weights
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
Transferred 437/437 items from pretrained weights
Freezing layer 'model.28.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 947.4±399.8 MB/s, size: 768.0 KB)
train: Scanning /kaggle/working/ds1/labels/train.cache... 4900 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4900/4900 856.3Mit/s 0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 932.5±163.4 MB/s, size: 758.9 KB)
val: Scanning /kaggle/working/ds1/labels/val.cache... 1225 images, 9 backgrounds, 0 corr

In [ ]:
from ultralytics import YOLO
import ultralytics.data.base as base, cv2, numpy as np

def imread_4ch(p, flags=cv2.IMREAD_COLOR):
    img = cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_UNCHANGED)
    return np.ascontiguousarray(img[:, :, [3, 0, 1, 2]])
base.imread = imread_4ch

m = YOLO('/kaggle/working/runs/detect/human_4ch_40ep/weights/best.pt')
for conf in (0.10, 0.15, 0.25, 0.35):
    r = m.val(data='/kaggle/working/ds1_4ch.yaml', imgsz=1024, batch=16,
              conf=conf, device=0, workers=2, plots=False, verbose=False)
    print(f'conf={conf:.2f}  P={r.box.mp:.3f}  R={r.box.mr:.3f}')